In [1]:
import pandas as pd

In [22]:
%%writefile stockFunctions.py
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

def rmsemape(y_true, y_pred):
    """Calculates and prints RMSE and MAPE."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"RMSE-Testset: {rmse}")
    print(f"maPe-Testset: {mape}")
    return rmse, mape

def graph(y_true, y_pred, label_true, label_pred, title, xlabel, ylabel):
    """Plots two series (actual and predicted) with given labels and title."""
    plt.figure(figsize=(10, 6))
    plt.plot(y_true, label=label_true)
    plt.plot(y_pred, label=label_pred)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.show()

def conversionSingle(data_array, column_names):
    """Converts a numpy array to a pandas DataFrame with specified column names."""
    if isinstance(data_array, pd.DataFrame):
        return data_array
    elif isinstance(data_array, np.ndarray):
        if data_array.ndim == 1:
            data_array = data_array.reshape(-1, 1)
        return pd.DataFrame(data_array, columns=column_names)
    else:
        try:
            data_array = np.array(data_array)
            if data_array.ndim == 1:
                data_array = data_array.reshape(-1, 1)
            return pd.DataFrame(data_array, columns=column_names)
        except Exception as e:
            print(f"Error in conversionSingle: Could not convert input to DataFrame. {e}")
            return None

Writing stockFunctions.py


In [23]:
!pip install nsepy
from nsepy import get_history as gh
import datetime as dt

In [8]:
start = dt.datetime(2021,6,1)
end = dt.datetime(2021,7,30)
try:
    stk_data = gh(symbol='TATACOFFEE',start=start,end=end)
except Exception as e:
    print(f"An error occurred while fetching data using nsepy: {e}")
    print("This often indicates an issue with the nsepy library's connection to the NSE website, possibly due to SSL/TLS problems or website changes.")
    print("Please consider trying again later, or exploring alternative methods/libraries to fetch NSE data.")
    stk_data = None # Set to None or an empty DataFrame if data fetching fails

An error occurred while fetching data using nsepy: HTTPSConnectionPool(host='www1.nseindia.com', port=443): Max retries exceeded with url: /products/dynaContent/common/productsSymbolMapping.jsp?dataType=PRICEVOLUMEDELIVERABLE&segmentLink=3&dateRange=&symbol=TATACOFFEE&series=EQ&symbolCount=1&fromDate=01-06-2021&toDate=30-07-2021 (Caused by SSLError(SSLError(1, '[SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1010)')))
This often indicates an issue with the nsepy library's connection to the NSE website, possibly due to SSL/TLS problems or website changes.
Please consider trying again later, or exploring alternative methods/libraries to fetch NSE data.


In [11]:
if stk_data is not None:
    stk_data=stk_data[["Open","High","Low","Close"]]
    stk_data.to_csv("Tatacoffee13_21.csv")
    print("Stock data processed and saved to Tatacoffee13_21.csv")
else:
    print("stk_data is None. Cannot process or save data.")

stk_data is None. Cannot process or save data.


In [12]:
column="Close"

In [14]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()

if stk_data is not None:
    data1 = Ms.fit_transform(stk_data[[column]])
    print("Len:", data1.shape)
else:
    print("stk_data is None. Cannot perform MinMaxScaler transformation.")
    data1 = None

stk_data is None. Cannot perform MinMaxScaler transformation.


In [16]:
if data1 is not None:
    training_size = round(len(data1 ) * 0.95)
    print(training_size)
    X_train=data1[:training_size]
    X_test=data1[training_size:]
    print("X_train length:",X_train.shape)
    print("X_test length:",X_test.shape)
    y_train=data1[:training_size]
    y_test=data1[training_size:]
    print("y_train length:",y_train.shape)
    print("y_test length:",y_test.shape)
else:
    print("data1 is None. Cannot split data into training and test sets.")

data1 is None. Cannot split data into training and test sets.


In [17]:
import warnings
warnings.filterwarnings("ignore")

In [26]:
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from stockFunctions import rmsemape

trends=['n','t','c','ct']
orders=[(0,0,1),(0,0,2)]
trendslist = []


if 'X_train' in locals() and X_train is not None:
    for td in trends:
        print(td)
        trendslist.append(td)
        model = ARIMA(X_train, order=(0,0,10),trend=td,)
        model_fit = model.fit()

        y_pred= model_fit.predict(len(X_train), len(data1)-1)
        print(y_pred)
        mse=mean_squared_error(y_test,y_pred,squared=False)
        print("Trend={}".format(td))
        rmsemape(y_test,y_pred)
        print("************")
else:
    print("Skipping ARIMA model training: X_train (or data1) is not available due to previous errors.")


Skipping ARIMA model training: X_train (or data1) is not available due to previous errors.


In [27]:
if 'y_pred' in locals() and y_pred is not None:
    print(len(y_pred))
else:
    print("y_pred is not defined because ARIMA model training was skipped.")

y_pred is not defined because ARIMA model training was skipped.


In [30]:
i=1
td="c"

if 'X_train' in locals() and X_train is not None and \
   'data1' in locals() and data1 is not None and \
   'y_test' in locals() and y_test is not None:
    model = ARIMA(X_train, order=(0,0,30),trend=td)
    model_fit = model.fit()
    y_pred= model_fit.predict(len(X_train), len(data1)-1)
    print(y_pred)
    from sklearn.metrics import r2_score
    mse=mean_squared_error(y_test,y_pred,squared=False)
    from stockFunctions import rmsemape
    print("Trend={}".format(td))
    rmsemape(y_test,y_pred)
    print("************")
else:
    print("Skipping ARIMA model training: X_train, data1, or y_test is not available due to previous errors.")
    y_pred = None # Ensure y_pred is defined as None if training is skipped

Skipping ARIMA model training: X_train, data1, or y_test is not available due to previous errors.


In [34]:
from stockFunctions import graph

if 'y_test' in locals() and y_test is not None and \
   'y_pred' in locals() and y_pred is not None:
    graph(y_test,y_pred,"Actual","Predicted","TataCoffee-Close-MA-Norm","Days","Prices")
else:
    print("Cannot plot: y_test or y_pred is not available because the model training was skipped.")

Cannot plot: y_test or y_pred is not available because the model training was skipped.


In [36]:
if data1 is not None:
    print(len(data1))
else:
    print("data1 is None. Cannot get its length because stock data fetching failed.")

data1 is None. Cannot get its length because stock data fetching failed.


In [40]:
from stockFunctions import conversionSingle

if 'y_test' in locals() and y_test is not None:
    aTestNormTable=conversionSingle(y_test,[column])
    actual_stock_price_test_ori=Ms.inverse_transform(aTestNormTable)
    actual_stock_price_test_oriA=conversionSingle(actual_stock_price_test_ori,[column])
    print("Original scale test data converted.")
else:
    print("y_test is not available. Skipping conversion to original scale.")
    aTestNormTable = None
    actual_stock_price_test_ori = None
    actual_stock_price_test_oriA = None

y_test is not available. Skipping conversion to original scale.


In [43]:
from stockFunctions import conversionSingle

if 'y_pred' in locals() and y_pred is not None:
    pTestNormTable=conversionSingle(y_pred,[column])

    if hasattr(Ms, 'scale_') and Ms.scale_ is not None:
        predicted_stock_price_test_ori=Ms.inverse_transform(pTestNormTable)
        predicted_stock_price_test_oriP=conversionSingle(predicted_stock_price_test_ori,[column])
        print("Predicted stock prices converted to original scale.")
    else:
        print("MinMaxScaler (Ms) was not fitted. Skipping conversion of predicted prices.")
        pTestNormTable = None
        predicted_stock_price_test_ori = None
        predicted_stock_price_test_oriP = None
else:
    print("y_pred is not available. Skipping conversion of predicted prices to original scale.")
    pTestNormTable = None
    predicted_stock_price_test_ori = None
    predicted_stock_price_test_oriP = None

y_pred is not available. Skipping conversion of predicted prices to original scale.


In [45]:
from stockFunctions import graph

if 'actual_stock_price_test_oriA' in locals() and actual_stock_price_test_oriA is not None and \
   'predicted_stock_price_test_oriP' in locals() and predicted_stock_price_test_oriP is not None:
    graph(actual_stock_price_test_oriA,predicted_stock_price_test_oriP,"Actual","Predicted","TataCoffee-Close-MA-Ori","Days","Prices")
else:
    print("Cannot plot original scale prices: actual_stock_price_test_oriA or predicted_stock_price_test_oriP is not available.")

Cannot plot original scale prices: actual_stock_price_test_oriA or predicted_stock_price_test_oriP is not available.


In [47]:
from stockFunctions import rmsemape

if 'actual_stock_price_test_oriA' in locals() and actual_stock_price_test_oriA is not None and \
   'predicted_stock_price_test_oriP' in locals() and predicted_stock_price_test_oriP is not None:
    rmsemape(actual_stock_price_test_oriA,predicted_stock_price_test_oriP)
else:
    print("Cannot calculate RMSE/MAPE: original scale actual or predicted prices are not available.")

Cannot calculate RMSE/MAPE: original scale actual or predicted prices are not available.


In [49]:
if 'model_fit' in locals() and model_fit is not None and \
   'data1' in locals() and data1 is not None:
    forecast=model_fit.predict(len(data1), len(data1))
    print("Forecast generated.")
else:
    print("Cannot generate forecast: model_fit or data1 is not available (ARIMA training was likely skipped).")
    forecast = None

Cannot generate forecast: model_fit or data1 is not available (ARIMA training was likely skipped).


In [ ]:
forecast

In [50]:
from stockFunctions import conversionSingle

forecast_stock_price_test_oriF = None # Initialize to None

if 'forecast' in locals() and forecast is not None and \
   hasattr(Ms, 'scale_') and Ms.scale_ is not None:
    fTestNormTable=conversionSingle(forecast,["Closefore"])

    if fTestNormTable is not None:
        forecast_stock_price_test_ori=Ms.inverse_transform(fTestNormTable)
        forecast_stock_price_test_oriF=conversionSingle(forecast_stock_price_test_ori,["Closefore"])
        print("Forecast converted to original scale.")
    else:
        print("Could not convert forecast to normal table. Check conversionSingle function.")
else:
    print("Cannot convert forecast to original scale: forecast is not available or MinMaxScaler (Ms) was not fitted.")

Cannot convert forecast to original scale: forecast is not available or MinMaxScaler (Ms) was not fitted.


In [51]:
forecast_stock_price_test_oriF

In [53]:
if 'forecast_stock_price_test_oriF' in locals() and forecast_stock_price_test_oriF is not None:
    forecast_stock_price_test_oriF.to_csv("CloseMA.csv",index=False)
    print("Forecast saved to CloseMA.csv")
else:
    print("Cannot save forecast to CSV: forecast_stock_price_test_oriF is not available.")

Cannot save forecast to CSV: forecast_stock_price_test_oriF is not available.
